# 📥 Bronze Loader — CSV → `jobs_harvested_bronze`

**Purpose**: Load daily CSV files (uploaded to DBFS) into the Bronze Delta table.
- Tracks loaded files in `bronze_load_audit` (no double loads)
- MERGE on `job_hash` → never inserts duplicate rows
- After running: trigger `02_silver_pipeline`

**DBFS Path**: Upload CSV to `/FileStore/jobs_daily/`

In [0]:
import uuid
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import col, lit, current_timestamp, when
from delta.tables import DeltaTable

spark = SparkSession.builder.getOrCreate()

CATALOG        = "jobs_automation_db"
SCHEMA         = "default"
BRONZE_TABLE   = f"{CATALOG}.{SCHEMA}.jobs_harvested_bronze"
AUDIT_TABLE    = f"{CATALOG}.{SCHEMA}.bronze_load_audit"
DBFS_JOBS_DIR  = "/FileStore/jobs_daily/"

print(f"✅ Config loaded. Bronze table: {BRONZE_TABLE}")

In [0]:
# Get all CSV files in DBFS directory
all_files = [f.path for f in dbutils.fs.ls(DBFS_JOBS_DIR) if f.name.endswith('.csv')]
print(f"📁 Total CSV files in {DBFS_JOBS_DIR}: {len(all_files)}")

# Get already-loaded files from audit table
try:
    loaded_files = set(
        spark.sql(f"SELECT file_path FROM {AUDIT_TABLE} WHERE status = 'success'")
        .rdd.flatMap(lambda x: x).collect()
    )
except Exception:
    loaded_files = set()
    print("ℹ️  Audit table empty or not found — will load all files")

# Filter to unprocessed files
to_process = [f for f in all_files if f not in loaded_files]
print(f"🆕 New files to process: {len(to_process)}")
for f in to_process:
    print(f"   → {f}")

In [0]:
BRONZE_SCHEMA = StructType([
    StructField("id",                    StringType(),  True),
    StructField("job_hash",              StringType(),  True),
    StructField("fetch_date",            DateType(),    True),
    StructField("portal",               StringType(),  True),
    StructField("search_keyword",        StringType(),  True),
    StructField("job_title",             StringType(),  True),
    StructField("company_name",          StringType(),  True),
    StructField("location",             StringType(),  True),
    StructField("remote_type",           StringType(),  True),
    StructField("salary_range",          StringType(),  True),
    StructField("experience_years",      StringType(),  True),
    StructField("tech_stack",            StringType(),  True),
    StructField("posted_date",           StringType(),  True),
    StructField("job_description",       StringType(),  True),
    StructField("description_length",    IntegerType(), True),
    StructField("roles_responsibilities",StringType(),  True),
    StructField("requirements_section",  StringType(),  True),
    StructField("roles_summary",         StringType(),  True),
    StructField("apply_link",            StringType(),  True),
    StructField("easy_apply_link",       StringType(),  True),
    StructField("company_career_url",    StringType(),  True),
    StructField("company_website",       StringType(),  True),
    StructField("hr_email",             StringType(),  True),
    StructField("job_id",               StringType(),  True),
    StructField("visa_sponsorship",      BooleanType(), True),
    StructField("validation_score",      IntegerType(), True),
    StructField("validation_status",     StringType(),  True),
    StructField("ai_summary",            StringType(),  True),
    StructField("detail_fetched",        BooleanType(), True),
])
print("✅ Bronze schema defined")

In [0]:
from pyspark.sql.functions import to_date

total_merged = 0

for file_path in to_process:
    print(f"\n📂 Processing: {file_path}")
    rows_loaded = 0
    rows_merged = 0
    status = "failed"

    try:
        # Read CSV
        df = spark.read.option("header", True) \
                       .option("multiLine", True) \
                       .option("escape", '"') \
                       .option("encoding", "UTF-8") \
                       .csv(file_path)

        rows_loaded = df.count()
        print(f"   📊 Rows read: {rows_loaded}")

        if rows_loaded == 0:
            print("   ⚠️  Empty file, skipping")
            status = "success"
            continue

        # Cast columns to correct types
        df = df \
            .withColumn("fetch_date",          to_date(col("fetch_date"))) \
            .withColumn("description_length",  col("description_length").cast(IntegerType())) \
            .withColumn("validation_score",    col("validation_score").cast(IntegerType())) \
            .withColumn("visa_sponsorship",    col("visa_sponsorship").cast(BooleanType())) \
            .withColumn("detail_fetched",      col("detail_fetched").cast(BooleanType()))

        # Drop rows with no job_hash
        df = df.filter(col("job_hash").isNotNull() & (col("job_hash") != ""))

        # MERGE into bronze (no duplicates)
        bronze_delta = DeltaTable.forName(spark, BRONZE_TABLE)

        (
            bronze_delta.alias("target")
            .merge(
                df.alias("source"),
                "target.job_hash = source.job_hash"
            )
            .whenNotMatchedInsertAll()
            .execute()
        )

        # Count newly inserted rows
        rows_merged = df.count()
        total_merged += rows_merged
        status = "success"
        print(f"   ✅ MERGE complete. New rows inserted (approx): {rows_merged}")

    except Exception as e:
        print(f"   ❌ Error: {e}")
        status = "failed"

    # Write to audit table
    audit_record = [(
        str(uuid.uuid4()),
        file_path.split("/")[-1],
        file_path,
        datetime.now(),
        rows_loaded,
        rows_merged,
        status,
    )]
    audit_schema = StructType([
        StructField("audit_id",    StringType(),    False),
        StructField("file_name",   StringType(),    True),
        StructField("file_path",   StringType(),    True),
        StructField("loaded_at",   TimestampType(), True),
        StructField("rows_loaded", IntegerType(),   True),
        StructField("rows_merged", IntegerType(),   True),
        StructField("status",      StringType(),    True),
    ])
    audit_df = spark.createDataFrame(audit_record, schema=audit_schema)
    audit_df.write.format("delta").mode("append").saveAsTable(AUDIT_TABLE)

print(f"\n🎉 Bronze loading complete! Total new records merged: {total_merged}")

In [0]:
%sql
SELECT 
    fetch_date,
    portal,
    COUNT(*) AS total_jobs,
    COUNT(DISTINCT company_name) AS unique_companies,
    SUM(CASE WHEN validation_status = 'Pending' THEN 1 ELSE 0 END) AS pending_validation
FROM jobs_automation_db.default.jobs_harvested_bronze
GROUP BY fetch_date, portal
ORDER BY fetch_date DESC, total_jobs DESC
LIMIT 50